In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset
import torchvision.transforms.v2 as transforms
from tqdm.auto import tqdm
from torchvision import transforms
from torchvision.datasets import MNIST # Training dataset
from torchvision.utils import make_grid
from torch.utils.data import DataLoader
from typing import Tuple, List, Dict, Optional, Callable
from glob import glob
import os

In [ ]:
torch.cuda.is_available()

# Dataset


- Segmentar os pulmões nas imagens
- Utilizar como entrada da rede gerativa

In [3]:
RAW_DATA_FOLDER = '/home/arthur/Documentos/transformers/data/atm22/'
PROCESSED_DATA_FOLDER = '/home/arthur/Documentos/generativas/dgm-2024.2/projetos/PulmoNet/data/processed'

In [4]:
class lungCTData(Dataset):
    def __init__(self, mode: str, transform: Optional[Callable] = None):
        super().__init__()
        self.cts = sorted(glob(os.path.join(PROCESSED_DATA_FOLDER, mode, "imagesTr", "*.npz")))[:30000]
        self.labels = sorted(glob(os.path.join(PROCESSED_DATA_FOLDER, mode, "lungsTr", "*.npz")))[:30000]
        self.transform = transform
        assert len(self.cts) == len(self.labels)

    def __len__(self):
        return len(self.cts)
    
    def __getitem__(self, idx: int):
        '''
        Carregar, transformar e retornar o item 'i' do dataset
        Cada aquisição envolve 4 sequências presentes na dimensão de canais da imagem
        As sequências mapeam com a string RawMRIDataset.SEQUENCE
        '''
        ct_path = self.cts[idx]
        ct_labels_path = self.labels[idx]

        # Ler imagem usando a biblioteca SimpleITK, o objeto image contêm também metadados
        # print(f'Reading {ct_path} and {ct_labels_path}.......')
        image_npz = np.load(ct_path)
        lung_npz = np.load(ct_labels_path)

        ct = image_npz['arr_0']
        lung = lung_npz['arr_0']
        ct = torch.tensor(ct).to(torch.float32)
        ct = ct.unsqueeze(0)
        lung = torch.tensor(lung).to(torch.float32).unsqueeze(0)


        # Se uma função de transformada foi passada para o dataset, aplicá-la
        if self.transform is not None:
            ct = self.transform(ct)
        # Retornar a imagem e metadados
        return ct, lung

# Segmentação

In [ ]:
print('hello world')

# Modelo

In [6]:
def encoder_block(in_dim,out_dim):
  return nn.Sequential(nn.Conv2d(in_channels=in_dim, out_channels=out_dim, kernel_size=4, stride=2, padding=1, dilation=1, bias=True),
                        nn.BatchNorm2d(out_dim),
                        nn.LeakyReLU(0.2,inplace=True))

In [7]:
def decoder_block(in_dim,out_dim):
  return nn.Sequential(nn.ConvTranspose2d(in_channels=in_dim, out_channels=out_dim, kernel_size=4, stride=2, padding=1, output_padding=0, groups=1, bias=True, dilation=1),
                       nn.BatchNorm2d(out_dim),
                       nn.Dropout(p=0.5),
                       nn.ReLU(inplace=True))

In [8]:
class Generator(nn.Module):
  def __init__(self):
    super(Generator, self).__init__()
    self.conv1 = nn.Sequential(nn.Conv2d(in_channels=1, out_channels=64, kernel_size=4, stride=2, padding=1, dilation=1, bias=True),
                               nn.LeakyReLU(0.2,inplace=True))
    self.conv2 = encoder_block(in_dim=64,out_dim=128)
    self.conv3 = encoder_block(in_dim=128,out_dim=256)
    self.conv4 = encoder_block(in_dim=256,out_dim=512)
    self.conv5 = encoder_block(in_dim=512,out_dim=512)
    self.conv6 = encoder_block(in_dim=512,out_dim=512)
    self.conv7 = encoder_block(in_dim=512,out_dim=512)
    self.conv8 = encoder_block(in_dim=512,out_dim=512)
    self.deconv8 = decoder_block(in_dim=512,out_dim=512)
    self.deconv7 = decoder_block(in_dim=512,out_dim=512)
    self.deconv6 = decoder_block(in_dim=512,out_dim=512)
    self.deconv5 = decoder_block(in_dim=512,out_dim=512)
    self.deconv4 = decoder_block(in_dim=512,out_dim=256)
    self.deconv3 = decoder_block(in_dim=256,out_dim=128)
    self.deconv2 = decoder_block(in_dim=128,out_dim=64)
    self.deconv1 = nn.Sequential(nn.ConvTranspose2d(in_channels=64, out_channels=1, kernel_size=4, stride=2, padding=1, output_padding=0, groups=1, bias=True, dilation=1),
                       nn.Tanh())

  def forward(self,x):
    enc1 = self.conv1(x) #64
    enc2 = self.conv2(enc1) #128
    enc3 = self.conv3(enc2) #256
    enc4 = self.conv4(enc3) #512
    enc5 = self.conv5(enc4) #512
    enc6 = self.conv6(enc5) #512
    enc7 = self.conv7(enc6) #512
    enc8 = self.conv8(enc7) #512
    dec8 = self.deconv8(enc8)+enc7
    dec7 = self.deconv7(dec8)+enc6
    dec6 = self.deconv6(dec7)+enc5
    dec5 = self.deconv5(dec6)+enc4
    dec4 = self.deconv4(dec5)+enc3
    dec3 = self.deconv3(dec4)+enc2
    dec2 = self.deconv2(dec3)+enc1
    dec1 = self.deconv1(dec2)
    return dec1

  def get_gen(self):
      return self.gen

In [ ]:
a = Generator()
a(torch.tensor(np.random.random_sample(size=(5,1,256,256)).astype('float32')))

In [10]:
def block_discriminator(in_dim,out_dim):
  return nn.Sequential(nn.Conv2d(in_channels=in_dim, out_channels=out_dim, kernel_size=4, stride=2, padding=1, dilation=1, bias=True),
                       nn.LeakyReLU(0.2,inplace=True))

In [11]:
class Discriminator(nn.Module):
  def __init__(self):
      super(Discriminator, self).__init__()
      self.conv1 = block_discriminator(in_dim=1,out_dim=64)
      self.conv2 = block_discriminator(in_dim=64,out_dim=128)
      self.conv3 = block_discriminator(in_dim=128,out_dim=256)
      self.conv4 = nn.Sequential(nn.Conv2d(in_channels=256, out_channels=512, kernel_size=4, stride=1, padding=1, dilation=1, bias=True),
                       nn.LeakyReLU(0.2,inplace=True))
      self.conv5 = nn.Sequential(nn.Conv2d(in_channels=512, out_channels=1, kernel_size=4, stride=1, padding=1, dilation=1, bias=True),
                       nn.Sigmoid())

  def forward(self,x):
      x = self.conv1(x)
      x = self.conv2(x)
      x = self.conv3(x)
      x = self.conv4(x)
      out = self.conv5(x)
      return out


  def get_disc(self):
        return self.disc

In [ ]:
def verify_discriminator():
    dummy_disc = Discriminator()
    

# Loss e Métricas

In [12]:
def get_disc_loss(gen,disc,criterion,input_mask,input_img,device):
    gen_img = gen(input_mask).detach()
    ans_gen = disc(gen_img)
    gt_gen = torch.zeros_like(ans_gen)
    ans_real = disc(input_img)
    gt_real = torch.ones_like(ans_real)

    # Concatenando os vetores do output do discriminador das reais com as geradas
    x = torch.cat((ans_real.reshape(-1),ans_gen.reshape(-1)))
    # Concatenando os vetores dos labels reais das images reais com as geradas
    y = torch.cat((gt_real.reshape(-1),gt_gen.reshape(-1)))
    loss = criterion(x,y)
    #the regularization (l1 norm) is not important here: is independent of D
    return loss.mean()

def get_gen_loss(gen,disc,criterion,input_mask,input_img,regularization,device):
    gen_img = gen(input_mask).detach()
    ans_gen = disc(gen_img)
    #we want ans_gen close to 1: to trick the disc
    gt_gen = torch.ones_like(ans_gen)
    loss_pt1 = criterion(ans_gen,gt_gen).mean()
    loss_pt2 = torch.sum(torch.abs(input_img-gen_img),dim=(1,2)).mean()
    return loss_pt1+regularization*loss_pt2

# Treinamento

In [13]:
# Set your parameters
criterion = nn.BCELoss(0.2)
n_epochs = 50
display_step = 500
lr = 2e-4 #Valor utilizado no artigo
batch_size = 512

### DO NOT EDIT ###
device = 'cuda'

In [14]:
gen = Generator().to(device)
gen_opt = torch.optim.Adam(gen.parameters(), lr=lr, betas=(0.5, 0.999))
gen_scheduler = torch.optim.lr_scheduler.LinearLR(gen_opt, start_factor=1.0, end_factor=0.0, total_iters=50)
disc = Discriminator().to(device)
disc_opt = torch.optim.Adam(disc.parameters(), lr=lr, betas=(0.5, 0.999))
disc_scheduler = torch.optim.lr_scheduler.LinearLR(disc_opt, start_factor=1.0, end_factor=0.0, total_iters=50)

In [15]:
dataset_train = lungCTData('train')
dataset_test = lungCTData('train')

In [ ]:
len(dataset_train)

In [17]:
data_loader_train = DataLoader(dataset_train, batch_size=8, shuffle=True)
data_loader_test = DataLoader(dataset_test, batch_size=8, shuffle=True)

In [59]:
for img, lab in data_loader_test:
    input_mask_ref = lab[:1].to(device)
    input_img_ref = img[:1].to(device)
    break

In [ ]:
np.shape(input_mask_ref)

In [ ]:
# OPTIONAL PART

cur_step = 0
mean_generator_loss = 0
mean_discriminator_loss = 0
regularization = 10

#adicionei isso aqui para observar a evolucao do treino
loss_disc_save = []
loss_gen_save = []
for epoch in range(n_epochs):

    # Dataloader returns the batches
    for i , batch in tqdm(enumerate(data_loader_train)):
        input_img_batch = batch[0]
        input_mask_batch = batch[1]

        input_img = input_img_batch.to(device)
        input_mask = input_mask_batch.to(device)
        if i % 3 == 0:
            ### Update discriminator ###
            # Zero out the gradients before backpropagation
            disc_opt.zero_grad()

            # Calculate discriminator loss
            disc_loss = get_disc_loss(gen,disc,criterion,input_mask,input_img,device)

            # Update gradients
            disc_loss.backward(retain_graph=True)

            # Update optimizer
            disc_opt.step()

        ### Update generator ###
        #     Hint: This code will look a lot like the discriminator updates!
        #     These are the steps you will need to complete:
        #       1) Zero out the gradients.
        #       2) Calculate the generator loss, assigning it to gen_loss.
        #       3) Backprop through the generator: update the gradients and optimizer.
        #### START CODE HERE ####
        gen_opt.zero_grad()
        gen_loss = get_gen_loss(gen,disc,criterion,input_mask,input_img,regularization,device)
        gen_loss.backward(retain_graph=True)
        gen_opt.step()
        #### END CODE HERE ####


        # Keep track of the average discriminator loss
        mean_discriminator_loss += disc_loss.item() / display_step

        # Keep track of the average generator loss
        mean_generator_loss += gen_loss.item() / display_step

        ### Visualization code ###
        if cur_step % display_step == 0 and cur_step > 0:
            print(f"Step {cur_step}: Generator loss: {mean_generator_loss}, discriminator loss: {mean_discriminator_loss}")
            
            #adicionei aqui para ver a evolucao do treino
            loss_disc_save.append(mean_discriminator_loss)
            loss_gen_save.append(mean_generator_loss)
            
            mean_generator_loss = 0
            mean_discriminator_loss = 0
            with torch.no_grad():
                gen_img_ref = gen(input_mask_ref)
                ans_gen_ref = disc(gen_img_ref)
                fig, ax = plt.subplots(1,3,figsize=(18,4))
                ax.flat[0].imshow(input_img_ref[0,0,:,:].detach().cpu().numpy(),cmap='gray')
                ax.flat[1].imshow(input_mask_ref[0,0,:,:].detach().cpu().numpy(),cmap='gray')
                ax.flat[2].imshow(gen_img_ref[0,0,:,:].detach().cpu().numpy(),cmap='gray')
                ax.flat[0].set_title('Original')
                ax.flat[1].set_title('Mask')
                ax.flat[2].set_title('Generated')
                plt.savefig('example_generated_epoch_'+str(epoch)+'.png')
        cur_step += 1
        gen_scheduler.step()
        disc_scheduler.step()

In [ ]:
for i , batch in enumerate(data_loader_train):
    print(np.shape(batch))
    a = batch[0].shape
    print(a)
    break

In [53]:
def gen_sample(input_mask):
    return gen(input_mask)


for img, label in data_loader_test:
    gen_img = gen_sample(label.to(device))
    break

In [ ]:
plt.imshow(np.squeeze(gen_img[0].cpu().detach().numpy(), 0), cmap='gray')

In [ ]:
plt.imshow(np.squeeze(label[0].cpu().detach().numpy(), 0), cmap='gray')